# MRMS \u00d7 HydroFabric \u2192 dHBV \u2014 Notebook 3 of 4: Event Manifest, MRMS Download & Fractional Crosswalk

**This notebook builds everything needed before rainfall can be extracted:**
1. **Per-event manifest (gage-basin method).** For each of the HUC8's flood events,
   find its recording gage (nearest gage downstream by network hops) and that
   gage's upstream contributing catchments \u2014 the confirmed, correct catchment
   association rule (footprint-radius buffering was tried and rejected: the
   `footprint_radius_km` field is report-scatter, not rain extent, and collapses
   to ~1 catchment per storm).
2. **Unique MRMS timestamp list.** Every event gets a 5-day window (midpoint
   \u00b1 2.5 days) at 2-min resolution; overlapping events share timestamps, so we
   download the UNION once.
3. **MRMS download**, using `mrms_bbox_downloader.py`'s `build_store()` \u2014
   file-level parallel download, in-memory GRIB decode, bbox subset, resumable
   day-shard NetCDF output. This is the confirmed production downloader (not the
   hand-rolled per-file loop from earlier drafts).
4. **Fractional (area-weighted) MRMS\u2192catchment crosswalk**, restricted to just
   the gage-basin catchments from step 1 \u2014 `fraction_inside` = area(MRMS cell
   \u2229 catchment) / area(MRMS cell), the input to area-weighted rainfall.

**Not in this notebook:** turning downloaded rainfall into per-catchment
2-min \u2192 15-min depth series and exporting NetCDF forcing \u2014 that's
**Notebook 4** (spatial area-weighting, then temporal aggregation, kept
separate since it's a materially different kind of work: reading GRIB/NetCDF
data instead of building spatial reference tables).

**Requires** `mrms_bbox_downloader.py` in the working directory.

## 0. Load Notebook 1 & 2 outputs

In [3]:
import os
import sys
import warnings
from pathlib import Path


import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import pyproj

pyproj.network.set_network_enabled(False)
warnings.filterwarnings("ignore")

# --- Notebook 1 outputs ---
NB1_DIR = Path("/projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess")
assert NB1_DIR.exists(), f"❌ {NB1_DIR} not found — run Notebook 1 first."

catchments_master = gpd.read_parquet(NB1_DIR / "catchments_master.parquet")
network           = pd.read_parquet(NB1_DIR / "network.parquet")
flowpaths         = gpd.read_parquet(NB1_DIR / "flowpaths.parquet")
nexus             = gpd.read_parquet(NB1_DIR / "nexus.parquet")
target_crs = catchments_master.crs

# --- Notebook 2 outputs (multi-HUC8 path) ---
NB2_DIR = Path("/projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess")
assert NB2_DIR.exists(), f"❌ {NB2_DIR} not found — run Notebook 2 first."

selected_huc8s = gpd.read_parquet(NB2_DIR / "selected_huc8s_multi.parquet")

# dtype forced to str -- otherwise pandas infers huc8_code as int64 and
# silently drops leading zeros (e.g. "01100005" -> 1100005)
events_multi = pd.read_csv(NB2_DIR / "events_multi.csv", dtype={"huc8_code": str})
events_multi["huc8_code"] = events_multi["huc8_code"].str.zfill(8)

huc8_codes = selected_huc8s["huc8_code"].tolist()

# --- CONUS crosswalk from Notebook 2 (still shared across all HUC8s) ---
CACHE = Path("mrms_crosswalk_cache")
COMBINED = CACHE / "mrms_hf_crosswalk_conus.parquet"
assert COMBINED.exists(), f"❌ {COMBINED} not found — run Notebook 2's Step 2 first."
crosswalk = pd.read_parquet(COMBINED)

# --- Rebuild the routing graph G (shared hydrofabric across all HUC8s) ---
edges = network.dropna(subset=["id", "toid"]).copy()
edges = edges[(edges["id"].astype(str).str.strip() != "") &
              (edges["toid"].astype(str).str.strip() != "")]
G = nx.from_pandas_edgelist(edges, source="id", target="toid", create_using=nx.DiGraph())

print(f"HUC8s: {huc8_codes}")
print(f"events_multi: {len(events_multi):,} rows across {events_multi['huc8_code'].nunique()} HUC8(s)")
print(f"unique huc8_code values: {sorted(events_multi['huc8_code'].unique())}")
print(f"crosswalk (CONUS, center-based): {len(crosswalk):,} rows")
print(f"G: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

HUC8s: ['03020201']
events_multi: 5,388 rows across 1 HUC8(s)
unique huc8_code values: ['03020201']
crosswalk (CONUS, center-based): 8,647,251 rows
G: 1,233,091 nodes, 1,233,090 edges


In [2]:
# new_events = pd.read_csv("upper_neuse_events_usgs.csv").rename(columns={
#     "event_id": "storm_index",
#     "BEGIN_DATE_TIME": "begin",
#     "END_DATE_TIME": "end",
# })

# # Make storm_index values distinguishable from the old cluster_id-based ones
# new_events["storm_index"] = "usgs_" + new_events["storm_index"].astype(str)

# # Register any gage not already snapped to the network
# missing = new_events.loc[~new_events["STAID"].isin(gages_indexed["STAID"]),
#                           ["STAID", "gage_lat", "gage_lon"]].drop_duplicates()
# if len(missing):
#     g_gdf = gpd.GeoDataFrame(
#         missing, geometry=gpd.points_from_xy(missing["gage_lon"], missing["gage_lat"]),
#         crs="EPSG:4326").to_crs(target_crs)
#     snapped = gpd.sjoin_nearest(
#         g_gdf, flowpaths[["id", "geometry"]], how="left", distance_col="gage_snap_dist_m"
#     ).rename(columns={"id": "gage_flowpath_id", "gage_lat": "LAT_GAGE", "gage_lon": "LNG_GAGE"})
#     gages_indexed = pd.concat([gages_indexed, snapped.drop(columns="geometry")], ignore_index=True)

# # Build event_catchment_windows directly from the KNOWN gage (skip the discovery loop)
# staid_to_fp = gages_indexed.drop_duplicates("STAID").set_index("STAID")["gage_flowpath_id"]
# new_events["gage_flowpath_id"] = new_events["STAID"].map(staid_to_fp)

# Section 1 \u2014 Per-event manifest (gage-basin method)

For each storm: find all gages **downstream** of it (`nx.descendants`), rank by
fewest network hops, keep the nearest \u2014 that's the **recording gage**. That
gage's **upstream contributing catchments** (`nx.ancestors`) are the catchments
associated with the event (cached per gage so repeated gages aren't recomputed).
Each event also gets a 5-day window centered on its midpoint.

**Method B (storm-footprint buffering by `footprint_radius_km`) is deliberately
not used** \u2014 that field is report-scatter (median \u2248 0.3 km, smaller than one
MRMS cell), so buffering collapses each event to ~1 catchment. Gage-basin is
the confirmed-correct method.

In [6]:
WINDOW_DAYS = 5.0
HALF = pd.to_timedelta(WINDOW_DAYS / 2, unit="D")

fp_to_divide = (
    network[["id", "divide_id"]].dropna()
    .assign(id=lambda d: d["id"].astype(str))
    .drop_duplicates(subset="id").set_index("id")["divide_id"]
)

unique_gages = events_multi[["STAID", "gage_lat", "gage_lon"]].drop_duplicates(subset="STAID")
g_gdf = gpd.GeoDataFrame(
    unique_gages,
    geometry=gpd.points_from_xy(unique_gages["gage_lon"], unique_gages["gage_lat"]),
    crs="EPSG:4326"
).to_crs(target_crs)
gages_indexed = gpd.sjoin_nearest(
    g_gdf, flowpaths[["id", "geometry"]], how="left", distance_col="gage_snap_dist_m"
).rename(columns={"id": "gage_flowpath_id"}).drop(columns="geometry").reset_index(drop=True)
print(f"Snapped {len(gages_indexed):,} unique gages")

gage_upstream_cats = {}
def upstream_cats_of(gfp):
    if gfp in gage_upstream_cats:
        return gage_upstream_cats[gfp]
    if gfp not in G:
        gage_upstream_cats[gfp] = set()
        return set()
    up = nx.ancestors(G, gfp); up.add(gfp)
    up_wb = {str(n) for n in up if str(n).startswith("wb-")}
    cats = set(fp_to_divide[fp_to_divide.index.isin(up_wb)].astype(str))
    gage_upstream_cats[gfp] = cats
    return cats

staid_to_fp = gages_indexed.drop_duplicates("STAID").set_index("STAID")["gage_flowpath_id"]

# ev = events_multi.copy()
# ev["storm_index"] = ev["huc8_code"] + "_" + ev["event_id"].astype(str)
# ev["begin_time"] = pd.to_datetime(ev["BEGIN_DATE_TIME"], errors="coerce", utc=True)
# ev["end_time"]   = pd.to_datetime(ev["END_DATE_TIME"],   errors="coerce", utc=True)
# ev["midpoint"]   = ev["begin_time"] + (ev["end_time"] - ev["begin_time"]) / 2
# ev["win_start"]  = ev["midpoint"] - HALF
# ev["win_end"]    = ev["midpoint"] + HALF
# ev["gage_flowpath_id"] = ev["STAID"].map(staid_to_fp)




ev = events_multi.copy()
# storm_index is already unique/huc8-prefixed from Notebook 2 -- don't rebuild it
ev["begin_time"] = pd.to_datetime(ev["begin"], errors="coerce", utc=True)
ev["end_time"]   = pd.to_datetime(ev["end"],   errors="coerce", utc=True)
ev["midpoint"]   = ev["begin_time"] + (ev["end_time"] - ev["begin_time"]) / 2
ev["win_start"]  = ev["midpoint"] - HALF
ev["win_end"]    = ev["midpoint"] + HALF
ev["gage_flowpath_id"] = ev["STAID"].map(staid_to_fp)

rows, no_gage = [], 0
for _, s in ev.iterrows():
    cats = upstream_cats_of(s["gage_flowpath_id"])
    if not cats:
        no_gage += 1
        continue
    rows.append({
        "storm_index": s["storm_index"], "huc8_code": s["huc8_code"],
        "recording_gage_STAID": s["STAID"], "gage_flowpath_id": s["gage_flowpath_id"],
        "n_catchments": len(cats), "midpoint": s["midpoint"],
        "win_start": s["win_start"], "win_end": s["win_end"],
        "_divide_ids": sorted(cats),
    })
manifest = pd.DataFrame(rows)
print(f"Resolved: {len(manifest):,} (unresolved: {no_gage})")

event_catchment_windows = (
    manifest.explode("_divide_ids").rename(columns={"_divide_ids": "divide_id"})
    .dropna(subset=["divide_id"])
    [["storm_index", "huc8_code", "recording_gage_STAID", "divide_id",
      "midpoint", "win_start", "win_end"]].reset_index(drop=True)
)
manifest_out = manifest.drop(columns="_divide_ids")

Snapped 12 unique gages
Resolved: 5,388 (unresolved: 0)


In [7]:
from mrms_bbox_downloader import bbox_from_huc8

MRMS_FREQ = "2min"
per_huc8_results = {}
for code in manifest_out["huc8_code"].unique():
    sub = manifest_out[manifest_out["huc8_code"] == code].copy()
    sub["win_start"] = pd.to_datetime(sub["win_start"], utc=True).dt.floor(MRMS_FREQ)
    sub["win_end"]   = pd.to_datetime(sub["win_end"],   utc=True).dt.ceil(MRMS_FREQ)

    windows = event_catchment_windows[event_catchment_windows["huc8_code"] == code]
    touched = catchments_master[catchments_master["divide_id"].isin(windows["divide_id"])]
    bbox = bbox_from_huc8(touched, margin_deg=0.1)

    parts = [pd.date_range(r["win_start"], r["win_end"], freq=MRMS_FREQ) for _, r in sub.iterrows()]
    if parts:
        unique_times = pd.DatetimeIndex(parts[0].append(parts[1:]) if len(parts) > 1 else parts[0]).unique().sort_values()
    else:
        unique_times = pd.DatetimeIndex([])
    if unique_times.tz is not None:
        unique_times = unique_times.tz_localize(None)

    per_huc8_results[code] = {"bbox": bbox, "unique_mrms_times": unique_times}
    print(f"[{code}] bbox={bbox} | {len(unique_times):,} timestamps")

import pickle
with open("multi_huc8_download_state.pkl", "wb") as f:
    pickle.dump(per_huc8_results, f)
print(f"\nSaved -> multi_huc8_download_state.pkl ({len(per_huc8_results)} HUC8s)")

[03020201] bbox=(-79.32831473786068, 35.409702227304045, -77.9877053408712, 36.4992199366684) | 1,266,602 timestamps

Saved -> multi_huc8_download_state.pkl (1 HUC8s)


In [9]:
# check = manifest_out.copy()
# check["window_days"] = (pd.to_datetime(check["win_end"]) - pd.to_datetime(check["win_start"])).dt.total_seconds() / 86400
# print(check.sort_values("window_days", ascending=False)[
#     ["storm_index", "huc8_code", "recording_gage_STAID", "win_start", "win_end", "window_days"]
# ].head(20))
# print("\nExpected: every row should be ~5.0 days.")
# print(f"Max window_days: {check['window_days'].max()}")
# print(f"Min window_days: {check['window_days'].min()}")

        storm_index huc8_code  recording_gage_STAID                 win_start  \
72088  15070102_851  15070102               9513860 2025-09-25 13:07:30+00:00   
0        01100005_1  01100005               1197000 2020-10-15 13:22:30+00:00   
1        01100005_2  01100005               1197000 2020-10-28 10:07:30+00:00   
2        01100005_3  01100005               1197000 2020-10-31 12:07:30+00:00   
3        01100005_4  01100005               1197000 2020-11-14 19:00:00+00:00   
4        01100005_5  01100005               1197000 2020-11-25 03:07:30+00:00   
5        01100005_6  01100005               1197000 2020-11-29 12:45:00+00:00   
6        01100005_7  01100005               1197000 2020-12-04 00:30:00+00:00   
7        01100005_8  01100005               1197000 2020-12-05 12:22:30+00:00   
8        01100005_9  01100005               1197000 2020-12-12 04:07:30+00:00   
9       01100005_10  01100005               1197000 2020-12-20 12:52:30+00:00   
10      01100005_11  0110000

In [10]:
# date_check = events_multi.copy()
# date_check["begin_dt"] = pd.to_datetime(date_check["begin"], errors="coerce", utc=True)
# print(date_check.groupby("huc8_code")["begin_dt"].agg(["min", "max", "count"]))

                                min                       max  count
huc8_code                                                           
01100005  2020-10-07 17:45:00+00:00 2025-09-25 21:30:00+00:00   5255
02030103  2020-10-01 00:15:00+00:00 2025-09-29 13:45:00+00:00  12354
02030105  2020-10-09 03:30:00+00:00 2025-09-30 18:15:00+00:00   8550
02040202  2020-10-01 01:00:00+00:00 2025-09-28 05:45:00+00:00   7761
02040205  2020-10-02 01:45:00+00:00 2025-09-30 08:45:00+00:00  10049
02070010  2020-10-01 13:45:00+00:00 2025-09-29 13:00:00+00:00   7982
03020201  2020-10-02 08:30:00+00:00 2025-09-30 15:15:00+00:00   5388
03030002  2020-10-01 01:00:00+00:00 2025-09-30 19:45:00+00:00   6782
05010009  2020-10-19 13:30:00+00:00 2025-09-23 00:15:00+00:00    540
05020002  2020-10-16 05:45:00+00:00 2025-09-24 13:30:00+00:00    954
05020003  2020-10-29 17:15:00+00:00 2025-09-25 05:15:00+00:00    363
05020005  2020-10-05 03:00:00+00:00 2025-09-25 14:30:00+00:00   1219
05030101  2020-10-19 14:45:00+00:0

In [8]:
total_unique_per_huc8 = sum(len(v["unique_mrms_times"]) for v in per_huc8_results.values())
all_times = pd.DatetimeIndex(sorted(set().union(*[set(v["unique_mrms_times"]) for v in per_huc8_results.values()])))
print(f"Sum of per-HUC8 timestamp counts: {total_unique_per_huc8:,}")
print(f"TRUE unique fetches needed (union across all 15 HUC8s): {len(all_times):,}")
print(f"Overlap savings from build_multi_store(): {1 - len(all_times)/total_unique_per_huc8:.1%}")

Sum of per-HUC8 timestamp counts: 1,266,602
TRUE unique fetches needed (union across all 15 HUC8s): 1,266,602
Overlap savings from build_multi_store(): 0.0%


## 1.1 Export a gage/event CSV (gage, start, end, tag)

One row per (event, recording gage), using each event's 5-day window and the
event's `cluster_id` as the tag. A gage with multiple events gets multiple rows.

In [9]:
gage_event_table = (
    manifest_out
    .assign(
        gage=lambda d: "gage-" + d["recording_gage_STAID"].astype(str),
        start=lambda d: pd.to_datetime(d["win_start"]).dt.strftime("%Y-%m-%d"),
        end=lambda d: pd.to_datetime(d["win_end"]).dt.strftime("%Y-%m-%d"),
        tag=lambda d: d["storm_index"],
    )
    [["gage", "start", "end", "tag"]]
    .sort_values(["gage", "start"])
    .reset_index(drop=True)
)

GAGE_EVENT_CSV = Path("/projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess/gage_event_table.csv")
gage_event_table.to_csv(GAGE_EVENT_CSV, index=False)

print(f"rows: {len(gage_event_table):,} | unique gages: {gage_event_table['gage'].nunique():,}")
print(f"saved: {GAGE_EVENT_CSV.resolve()}")
gage_event_table.head()

rows: 5,388 | unique gages: 12
saved: /projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess/gage_event_table.csv


,gage,start,end,tag
0,gage-2085000,2020-10-01,2020-10-06,03020201_1
1,gage-2085000,2020-10-02,2020-10-07,03020201_2
2,gage-2085000,2020-10-03,2020-10-08,03020201_3
3,gage-2085000,2020-10-04,2020-10-09,03020201_4
4,gage-2085000,2020-10-05,2020-10-10,03020201_5


In [10]:
import pandas as pd

# Merge manifest with event storm_index
tmp = (
    manifest_out
    .assign(
        gage=lambda d: "gage-" + d["recording_gage_STAID"].astype(str),
        win_start_dt=lambda d: pd.to_datetime(d["win_start"]),
        win_end_dt=lambda d: pd.to_datetime(d["win_end"]),
    )
)

# For each gage, find earliest start, latest end, and storm_index of earliest storm
gage_event_table = (
    tmp.sort_values(["gage", "win_start_dt"])
    .groupby("gage", as_index=False)
    .agg(
        earliest_start=("win_start_dt", "min"),
        latest_end=("win_end_dt", "max"),
        tag=("storm_index", "first"),
    )
    .assign(
        start=lambda d: (d["earliest_start"] - pd.DateOffset(months=1)).dt.strftime("%Y-%m-%d"),
        end=lambda d: (d["latest_end"] + pd.DateOffset(months=1)).dt.strftime("%Y-%m-%d"),
    )
    [["gage", "start", "end", "tag"]]
    .sort_values("gage")
    .reset_index(drop=True)
)

GAGE_EVENT_CSV = Path("gage_event_table.csv")
gage_event_table.to_csv(GAGE_EVENT_CSV, index=False)

print(f"rows: {len(gage_event_table):,} | unique gages: {gage_event_table['gage'].nunique():,}")
print(f"saved: {GAGE_EVENT_CSV.resolve()}")
gage_event_table.head()

rows: 12 | unique gages: 12
saved: /projects/mhpi/leoglonz/sub_hourly/flash_preprocess/engine/forcing/mrms/gage_event_table.csv


,gage,start,end,tag
0,gage-2085000,2020-09-01,2025-11-02,03020201_1
1,gage-2085070,2020-09-08,2025-11-03,03020201_545
2,gage-2085500,2020-09-08,2025-10-10,03020201_1045
3,gage-2086500,2020-09-10,2025-10-11,03020201_1309
4,gage-2086624,2020-09-02,2025-09-15,03020201_1917


# Section 2 \u2014 Unique MRMS timestamp list

Each event window is expanded to 2-min steps, then we take the UNION across
all events \u2014 overlapping windows share timestamps, so a shared file only gets
downloaded once.

In [11]:
MRMS_FREQ = "2min"

mw = manifest.dropna(subset=["win_start", "win_end"]).copy()
mw["win_start"] = pd.to_datetime(mw["win_start"], utc=True).dt.floor(MRMS_FREQ)
mw["win_end"]   = pd.to_datetime(mw["win_end"],   utc=True).dt.ceil(MRMS_FREQ)

# expand every event window to 2-min steps -> long (event, time) table
rows = []
for _, r in mw.iterrows():
    times = pd.date_range(r["win_start"], r["win_end"], freq=MRMS_FREQ)
    rows.append(pd.DataFrame({"storm_index": r["storm_index"], "mrms_time": times}))

event_time_manifest = pd.concat(rows, ignore_index=True)

# the deduplicated set of timestamps to actually download
unique_mrms_times = (
    event_time_manifest["mrms_time"]
    .drop_duplicates().sort_values().reset_index(drop=True)
)

print(f"Events expanded            : {mw['storm_index'].nunique():,}")
print(f"(event, time) rows         : {len(event_time_manifest):,}")
print(f"UNIQUE timestamps to fetch : {len(unique_mrms_times):,}")
print(f"Dedup saving               : "
      f"{1 - len(unique_mrms_times)/len(event_time_manifest):.1%} fewer files")
print(f"Time span                  : {unique_mrms_times.min()} \u2192 {unique_mrms_times.max()}")


Events expanded            : 5,388
(event, time) rows         : 19,406,039
UNIQUE timestamps to fetch : 1,266,602
Dedup saving               : 93.5% fewer files
Time span                  : 2020-09-29 22:52:00+00:00 → 2025-10-03 07:00:00+00:00


In [13]:
from pathlib import Path
from mrms_bbox_downloader import bbox_from_huc8, build_store, consolidate

# bbox padded 0.1 deg around the selected HUC8 (lon/lat, EPSG:4326)
bbox = bbox_from_huc8(selected_huc8s, margin_deg=0.1)
print("bbox (lonmin, latmin, lonmax, latmax):", bbox)

# Absolute path -- pinning this matters. A relative path like "storm_precip_MRMS"
# resolves against whatever the notebook's current working directory happens to
# be, which can differ between kernel restarts / how you launched Jupyter.
MRMS_OUT_DIR = Path("/projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess").resolve()
SHARD_DIR = MRMS_OUT_DIR / "shards"

bbox (lonmin, latmin, lonmax, latmax): (-79.32866436499995, 35.0738186750001, -77.87203142200002, 36.49927568000008)


In [9]:
# import pickle
# from pathlib import Path

# CACHE = Path("nb3_download_state.pkl")
# with open(CACHE, "wb") as f:
#     pickle.dump({"bbox": bbox, "unique_mrms_times": unique_mrms_times}, f)
# print(f"cached -> {CACHE.resolve()}")

In [10]:
# import pickle
# from pathlib import Path

# with open("nb3_download_state.pkl", "rb") as f:
#     state = pickle.load(f)
# bbox, unique_mrms_times = state["bbox"], state["unique_mrms_times"]

In [11]:
# RESUME_FROM_CACHE = False  # flip to True after a kernel crash

# if RESUME_FROM_CACHE:
#     import pickle
#     with open("nb3_download_state.pkl", "rb") as f:
#         state = pickle.load(f)
#     bbox, unique_mrms_times = state["bbox"], state["unique_mrms_times"]
#     print("Restored bbox and unique_mrms_times from cache — skip ahead to the build_store() cell.")

In [12]:
# import pickle
# from pathlib import Path
# from mrms_bbox_downloader import bbox_from_huc8, build_store, consolidate

# CACHE = Path("nb3_download_state.pkl")
# with open(CACHE, "rb") as f:
#     state = pickle.load(f)
# bbox, unique_mrms_times = state["bbox"], state["unique_mrms_times"]

# MRMS_OUT_DIR = Path("storm_precip_MRMS").resolve()
# SHARD_DIR = MRMS_OUT_DIR / "shards"

# print(f"Restored: bbox={bbox}")
# print(f"Restored: {len(unique_mrms_times):,} timestamps "
#       f"({unique_mrms_times.min()} → {unique_mrms_times.max()})")
# print(f"MRMS_OUT_DIR: {MRMS_OUT_DIR}")
# print("Ready — skip straight to the build_store() cell.")

# Section 3 \u2014 Download MRMS PrecipRate via `mrms_bbox_downloader.py`

`build_store()` parallelizes over individual 2-min files (not days), decodes
each GRIB2 **in memory** (the full-CONUS grid is never written to disk), and
subsets to the HUC8 bounding box before writing one NetCDF **day shard** per
UTC day under `<out_dir>/shards/`. It's resumable: rerun to fill gaps, and
confirmed-missing timestamps (404 in both archives) are remembered and skipped
next time \u2014 so this cell is safe to interrupt and re-run.

In [15]:
import sys
!{sys.executable} -m pip install -q numpy pandas xarray cfgrib eccodes s3fs fsspec geopandas tqdm

In [16]:
import pickle
from pathlib import Path
from mrms_bbox_downloader import bbox_from_huc8, build_store, consolidate

# ---- restore cached state (skips re-deriving HUC8/crosswalk/hydrofabric) ----
CACHE = Path("/projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess/multi_huc8_download_state.pkl")
with open(CACHE, "rb") as f:
    state = pickle.load(f)
bbox, unique_mrms_times = state["bbox"], state["unique_mrms_times"]

MRMS_OUT_DIR = Path("/projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess").resolve()
SHARD_DIR = MRMS_OUT_DIR / "shards"


print(f"Restored bbox: {bbox}")
print(f"Restored {len(unique_mrms_times):,} timestamps "
      f"({unique_mrms_times.min()} → {unique_mrms_times.max()})")
print(f"MRMS_OUT_DIR: {MRMS_OUT_DIR}")


import pandas as pd

if getattr(unique_mrms_times, "tz", None) is not None or getattr(unique_mrms_times.dt, "tz", None) is not None:
    print("⚠️ unique_mrms_times was tz-aware -- stripping tz to match the tz-naive shard store.")
    unique_mrms_times = pd.DatetimeIndex(unique_mrms_times).tz_localize(None)

KeyError: 'bbox'

In [18]:
state["unique_mrms_times"]

KeyError: 'unique_mrms_times'

In [17]:
state

{'03020201': {'bbox': (-79.32831473786068,
   35.409702227304045,
   -77.9877053408712,
   36.4992199366684),
  'unique_mrms_times': DatetimeIndex(['2020-09-29 22:52:00', '2020-09-29 22:54:00',
                 '2020-09-29 22:56:00', '2020-09-29 22:58:00',
                 '2020-09-29 23:00:00', '2020-09-29 23:02:00',
                 '2020-09-29 23:04:00', '2020-09-29 23:06:00',
                 '2020-09-29 23:08:00', '2020-09-29 23:10:00',
                 ...
                 '2025-10-03 06:42:00', '2025-10-03 06:44:00',
                 '2025-10-03 06:46:00', '2025-10-03 06:48:00',
                 '2025-10-03 06:50:00', '2025-10-03 06:52:00',
                 '2025-10-03 06:54:00', '2025-10-03 06:56:00',
                 '2025-10-03 06:58:00', '2025-10-03 07:00:00'],
                dtype='datetime64[ns]', length=1266602, freq=None)}}

In [19]:
import pandas as pd
import xarray as xr

# How many of your target timestamps are actually covered by what's on disk?
covered = 0
for f in sorted(SHARD_DIR.glob("pr_*.nc")):
    with xr.open_dataset(f) as ds:
        covered += ds.sizes.get("time", 0)

expected_present = len(unique_mrms_times) - 2512  # known-missing count from your log
print(f"Timestamps on disk : {covered:,}")
print(f"Expected present   : {expected_present:,}")
print(f"Difference         : {expected_present - covered:,}")

Timestamps on disk : 0
Expected present   : 1,264,090
Difference         : 1,264,090


In [ ]:




import xarray as xr

# bbox and unique_mrms_times already restored from cache in the cell above --
# no need to recompute from selected_huc8.

# Pre-flight check: don't just count existing shard files -- actually try to
# open each one with xarray, the same way build_store()'s internal
# _read_existing() does. If a file is unreadable (truncated write, older/
# incompatible encoding, etc.), _read_existing() silently catches the
# exception and treats that day as if it had never been downloaded, which is
# what causes a "files exist on disk but everything re-downloads anyway"
# situation.
existing_shards = sorted(SHARD_DIR.glob("pr_*.nc")) if SHARD_DIR.exists() else []
existing_bytes = sum(f.stat().st_size for f in existing_shards)
print(f"Existing day shard files: {len(existing_shards):,} "
      f"({existing_bytes / 1e9:.2f} GB) in {SHARD_DIR}")

readable, broken = 0, []
n_times_found = 0
for f in existing_shards:
    try:
        with xr.open_dataset(f) as ds:
            n_times_found += ds.sizes.get("time", 0)
        readable += 1
    except Exception as e:
        broken.append((f.name, repr(e)))

print(f"Readable: {readable:,} / {len(existing_shards):,} "
      f"({n_times_found:,} timestamps total)")
if not existing_shards:
    print("  -> none found at this path. If you expected existing files, "
          "double check MRMS_OUT_DIR matches the folder used previously.")
elif broken:
    print(f"  ⚠️ {len(broken):,} file(s) failed to open -- this is almost "
          "certainly why build_store() thinks nothing is stored and "
          "re-downloads. First few failures:")
    for name, err in broken[:5]:
        print(f"    - {name}: {err}")
elif readable and n_times_found == 0:
    print("  ⚠️ Files open fine but contain 0 timestamps -- likely empty/"
          "placeholder shards from an interrupted run.")
else:
    print("  ✅ All existing shards are readable and contain data -- "
          "build_store() below should resume, not re-download.")

summary = build_store(
    unique_mrms_times,
    bbox,
    str(MRMS_OUT_DIR),
    max_workers=4,
    mask_negative=True,   # PrecipRate -3 ("no radar coverage") -> NaN
    use_aws=True,         # AWS noaa-mrms-pds first, Iowa State HTTPS fallback
    verbose=True,
)
print(summary)
print(f"\nDay shards: {SHARD_DIR.resolve()}")

Existing day shard files: 0 (0.00 GB) in /projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess/shards
Readable: 0 / 0 (0 timestamps total)
  -> none found at this path. If you expected existing files, double check MRMS_OUT_DIR matches the folder used previously.
1266602 timestamps across 1780 UTC days | 0 already stored, 1266602 to fetch -> /projects/mhpi/leoglonz/sub_hourly/data/_mrms_preprocess/shards


files:   0%|          | 0/1266602 [00:00<?, ?file/s]

In [17]:
# from pathlib import Path
# from mrms_bbox_downloader import bbox_from_huc8, build_store, consolidate

# # bbox padded 0.1 deg around the selected HUC8 (lon/lat, EPSG:4326)
# bbox = bbox_from_huc8(selected_huc8, margin_deg=0.1)
# print("bbox (lonmin, latmin, lonmax, latmax):", bbox)

# # Absolute path -- pinning this matters. A relative path like "storm_precip_MRMS"
# # resolves against whatever the notebook's current working directory happens to
# # be, which can differ between kernel restarts / how you launched Jupyter.
# MRMS_OUT_DIR = Path("storm_precip_MRMS").resolve()
# SHARD_DIR = MRMS_OUT_DIR / "shards"
# print("MRMS_OUT_DIR:", MRMS_OUT_DIR)

# # Safety net: build_store() compares these timestamps against tz-naive
# # np.datetime64 values already on disk. If unique_mrms_times is still
# # tz-aware (e.g. this cell is re-run without re-running the timestamp cell
# # above), the "already stored" check silently fails and everything
# # re-downloads. Strip tz here too, defensively, regardless of upstream state.
# if getattr(unique_mrms_times.dt, "tz", None) is not None:
#     print("⚠️ unique_mrms_times was tz-aware -- stripping tz to match the "
#           "tz-naive shard store.")
#     unique_mrms_times = unique_mrms_times.dt.tz_localize(None)

# # Pre-flight check: don't just count existing shard files -- actually try to
# # open each one with xarray, the same way build_store()'s internal
# # _read_existing() does. If a file is unreadable (truncated write, older/
# # incompatible encoding, etc.), _read_existing() silently catches the
# # exception and treats that day as if it had never been downloaded, which is
# # what causes a "files exist on disk but everything re-downloads anyway"
# # situation. Surfacing that here, before the download starts, turns a
# # confusing multi-hour re-download into an immediate, readable error list.
# import xarray as xr

# existing_shards = sorted(SHARD_DIR.glob("pr_*.nc")) if SHARD_DIR.exists() else []
# existing_bytes = sum(f.stat().st_size for f in existing_shards)
# print(f"Existing day shard files: {len(existing_shards):,} "
#       f"({existing_bytes / 1e9:.2f} GB) in {SHARD_DIR}")

# readable, broken = 0, []
# n_times_found = 0
# for f in existing_shards:
#     try:
#         with xr.open_dataset(f) as ds:
#             n_times_found += ds.sizes.get("time", 0)
#         readable += 1
#     except Exception as e:
#         broken.append((f.name, repr(e)))

# print(f"Readable: {readable:,} / {len(existing_shards):,} "
#       f"({n_times_found:,} timestamps total)")

# if not existing_shards:
#     print("  -> none found at this path. If you expected existing files, "
#           "double check MRMS_OUT_DIR matches the folder used previously.")
# elif broken:
#     print(f"  ⚠️ {len(broken):,} file(s) failed to open -- this is almost "
#           "certainly why build_store() thinks nothing is stored and "
#           "re-downloads. First few failures:")
#     for name, err in broken[:5]:
#         print(f"    - {name}: {err}")
# elif readable and n_times_found == 0:
#     print("  ⚠️ Files open fine but contain 0 timestamps -- likely empty/"
#           "placeholder shards from an interrupted run.")
# else:
#     print("  ✅ All existing shards are readable and contain data -- "
#           "build_store() below should resume, not re-download.")

# summary = build_store(
#     unique_mrms_times,
#     bbox,
#     str(MRMS_OUT_DIR),
#     max_workers=8,
#     mask_negative=True,   # PrecipRate -3 ("no radar coverage") -> NaN
#     use_aws=True,         # AWS noaa-mrms-pds first, Iowa State HTTPS fallback
#     verbose=True,
# )
# print(summary)
# print(f"\nDay shards: {SHARD_DIR.resolve()}")

### 3.1 Optional: consolidate into one NetCDF/Zarr store

Not required \u2014 Notebook 4 can read the per-day shards directly and only
needs to open the days each event's window touches. Consolidating everything
into one file is convenient for browsing but duplicates the data on disk.

In [ ]:
# consolidate(str(SHARD_DIR), MRMS_OUT_DIR + "/precip_rate.nc", fmt="netcdf")

# Section 4 \u2014 Fractional (area-weighted) MRMS\u2192catchment crosswalk

Restricted to just the gage-basin catchments from Section 1
(`event_catchment_windows['divide_id']`) \u2014 this is the same true-intersection
polygon math as the full-CONUS version, just scoped down so it runs in minutes
instead of hours. `fraction_inside` = area(MRMS cell \u2229 catchment) / area(MRMS
cell); weights are then normalized to sum to 1 per catchment.

In [23]:
# ============================================================
# 4.1 Narrow catchments_master + the center-based crosswalk down to the
#     gage-basin catchments (method-agnostic vs. the footprint-buffer
#     version in the original notebook -- just a different divide_id set).
# ============================================================
selected_divide_ids = event_catchment_windows["divide_id"].dropna().unique()
print(f"Gage-basin catchments (event-relevant): {len(selected_divide_ids):,}")

selected_catchments = catchments_master[
    catchments_master["divide_id"].isin(selected_divide_ids)
].copy()
print(f"selected_catchments rows: {len(selected_catchments):,}")
print(f"selected VPUs: {selected_catchments['vpuid'].nunique():,}")

selected_crosswalk = crosswalk[
    crosswalk["divide_id"].isin(selected_divide_ids)
].copy()
print(f"selected_crosswalk rows: {len(selected_crosswalk):,}")
print(f"unique MRMS cells (center-based): {selected_crosswalk['cell_id'].nunique():,}")

# STEP3_CACHE = CACHE / "step3_event_relevant"
# STEP3_CACHE.mkdir(exist_ok=True)

STEP3_CACHE = Path("event_mrms_outputs") / "step3_event_relevant"
STEP3_CACHE.mkdir(parents=True, exist_ok=True)

selected_catchments.to_parquet(STEP3_CACHE / "selected_catchments.parquet", index=False)

Gage-basin catchments (event-relevant): 277
selected_catchments rows: 277
selected VPUs: 1
selected_crosswalk rows: 2,119
unique MRMS cells (center-based): 2,119


In [24]:
# ============================================================
# 4.2 Build fractional crosswalk for selected (gage-basin) catchments only
#
# Inputs:  selected_catchments, selected_crosswalk, CACHE
# Outputs: selected_fractional_crosswalk (+ .parquet)
# ============================================================

import time
from shapely.geometry import box

# ------------------------------------------------------------
# MRMS grid settings
# ------------------------------------------------------------
MRMS_LON_MIN_EDGE = -130.0
MRMS_LAT_MAX_EDGE = 55.0
MRMS_RES = 0.01
MRMS_NLON = 7000
MRMS_NLAT = 3500
HALF = MRMS_RES / 2.0
AREA_CRS = 5070

# Neighbor padding around center-based MRMS cells
# 1 is usually enough; 2 is safer for edge cases.
PAD_CELLS = 2

FRAC_SELECTED_CACHE = STEP3_CACHE / "selected_fractional_crosswalk_cache"
FRAC_SELECTED_CACHE.mkdir(parents=True, exist_ok=True)
SELECTED_FRAC_FILE = STEP3_CACHE / "selected_fractional_crosswalk.parquet"


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def lon_from_col(col):
    return MRMS_LON_MIN_EDGE + HALF + col * MRMS_RES

def lat_from_row(row):
    return MRMS_LAT_MAX_EDGE - HALF - row * MRMS_RES

def cell_id_from_row_col(row, col):
    return row.astype(np.int64) * MRMS_NLON + col.astype(np.int64)

def build_mrms_cells_from_rows_cols(rows, cols):
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)

    lon = lon_from_col(cols)
    lat = lat_from_row(rows)

    gdf = gpd.GeoDataFrame(
        {
            "cell_id": cell_id_from_row_col(rows, cols),
            "row": rows,
            "col": cols,
            "lon": lon,
            "lat": lat,
            "geometry": [
                box(x - HALF, y - HALF, x + HALF, y + HALF)
                for x, y in zip(lon, lat)
            ]
        },
        geometry="geometry",
        crs="EPSG:4326"
    ).to_crs(AREA_CRS)

    gdf["cell_area_m2"] = gdf.geometry.area

    return gdf

def rows_cols_for_bbox(west, south, east, north, pad=2):
    i0 = int(np.floor((west - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) - pad
    i1 = int(np.ceil((east - MRMS_LON_MIN_EDGE - HALF) / MRMS_RES)) + pad

    j0 = int(np.floor((MRMS_LAT_MAX_EDGE - HALF - north) / MRMS_RES)) - pad
    j1 = int(np.ceil((MRMS_LAT_MAX_EDGE - HALF - south) / MRMS_RES)) + pad

    i0 = max(0, i0)
    i1 = min(MRMS_NLON - 1, i1)

    j0 = max(0, j0)
    j1 = min(MRMS_NLAT - 1, j1)

    if i1 < i0 or j1 < j0:
        return pd.DataFrame(columns=["row", "col"])

    cols = np.arange(i0, i1 + 1)
    rows = np.arange(j0, j1 + 1)

    COL, ROW = np.meshgrid(cols, rows)

    return pd.DataFrame({
        "row": ROW.ravel().astype(np.int32),
        "col": COL.ravel().astype(np.int32)
    })

def add_neighbor_cells(df, pad=2):
    base = df[["row", "col"]].drop_duplicates().copy()

    pieces = []

    for dr in range(-pad, pad + 1):
        for dc in range(-pad, pad + 1):
            tmp = base.copy()
            tmp["row"] = tmp["row"] + dr
            tmp["col"] = tmp["col"] + dc
            pieces.append(tmp)

    out = pd.concat(pieces, ignore_index=True)

    out = out[
        out["row"].between(0, MRMS_NLAT - 1) &
        out["col"].between(0, MRMS_NLON - 1)
    ]

    return out.drop_duplicates().reset_index(drop=True)

def intersection_area_chunked(candidates, chunk_size=150_000):
    areas = []

    for start in range(0, len(candidates), chunk_size):
        end = min(start + chunk_size, len(candidates))
        chunk = candidates.iloc[start:end].copy()

        catchment_geoms = gpd.GeoSeries(
            chunk["catchment_geometry"],
            crs=AREA_CRS,
            index=chunk.index
        )

        inter = chunk.geometry.intersection(catchment_geoms)
        areas.append(inter.area.to_numpy())

    return np.concatenate(areas)


# ------------------------------------------------------------
# Run by VPU to control memory
# ------------------------------------------------------------
parts = []
t0 = time.time()

vpus = sorted(selected_catchments["vpuid"].dropna().astype(str).unique())

print(f"VPUs to process: {len(vpus)}")
print(vpus)

for vpu in vpus:

    out_file = FRAC_SELECTED_CACHE / f"selected_fractional_crosswalk_vpu_{vpu}.parquet"

    if out_file.exists():
        df_cached = pd.read_parquet(out_file)
        parts.append(df_cached)
        print(f"{vpu}: cached ({len(df_cached):,} rows)")
        continue

    print(f"\nWorking on VPU {vpu}...")

    sub = selected_catchments[
        selected_catchments["vpuid"].astype(str) == vpu
    ].copy()

    if len(sub) == 0:
        print(f"{vpu}: no selected catchments")
        continue

    sub = sub.to_crs(AREA_CRS)

    try:
        sub["geometry"] = sub.geometry.make_valid()
    except Exception:
        sub["geometry"] = sub.geometry.buffer(0)

    divide_ids_vpu = sub["divide_id"].dropna().unique()

    # --------------------------------------------------------
    # Candidate cells from center-based crosswalk + neighbors
    # --------------------------------------------------------
    cw_vpu = selected_crosswalk[
        selected_crosswalk["divide_id"].isin(divide_ids_vpu)
    ][["divide_id", "cell_id", "row", "col"]].copy()

    candidate_cells = add_neighbor_cells(
        cw_vpu[["row", "col"]],
        pad=PAD_CELLS
    )

    # --------------------------------------------------------
    # Add cells for catchments with no center-based MRMS cell
    # --------------------------------------------------------
    catchments_with_center_cell = set(cw_vpu["divide_id"].dropna().unique())
    missing_center_catchments = sub[
        ~sub["divide_id"].isin(catchments_with_center_cell)
    ].copy()

    print(f"  selected catchments: {len(sub):,}")
    print(f"  catchments without center MRMS cell: {len(missing_center_catchments):,}")

    extra_cells = []

    if len(missing_center_catchments) > 0:
        missing_4326 = missing_center_catchments.to_crs(4326)

        for geom in missing_4326.geometry:
            west, south, east, north = geom.bounds
            extra_cells.append(
                rows_cols_for_bbox(west, south, east, north, pad=PAD_CELLS)
            )

    if len(extra_cells) > 0:
        extra_cells = pd.concat(extra_cells, ignore_index=True)
        candidate_cells = pd.concat(
            [candidate_cells, extra_cells],
            ignore_index=True
        )

    candidate_cells = (
        candidate_cells
        .drop_duplicates(subset=["row", "col"])
        .reset_index(drop=True)
    )

    print(f"  candidate MRMS cells: {len(candidate_cells):,}")

    if len(candidate_cells) == 0:
        print(f"{vpu}: no candidate cells")
        continue

    # --------------------------------------------------------
    # Build MRMS cell polygons
    # --------------------------------------------------------
    mrms_cells = build_mrms_cells_from_rows_cols(
        candidate_cells["row"].values,
        candidate_cells["col"].values
    )

    # --------------------------------------------------------
    # Spatial join: MRMS cell polygons intersect catchments
    # --------------------------------------------------------
    join_cols = [
        "divide_id",
        "vpuid",
        "nexus_id",
        "nexus_type",
        "is_terminal",
        "geometry"
    ]

    if "terminal_nexus_id" in sub.columns:
        join_cols.insert(-1, "terminal_nexus_id")

    join_cols = [c for c in join_cols if c in sub.columns]

    sub_join = sub[join_cols].copy()

    candidates = gpd.sjoin(
        mrms_cells,
        sub_join,
        how="inner",
        predicate="intersects"
    )

    if len(candidates) == 0:
        print(f"{vpu}: no intersecting cells")
        continue

    print(f"  candidate cell/catchment pairs: {len(candidates):,}")

    # Attach catchment geometry
    catchment_geom_lookup = sub_join.geometry.rename("catchment_geometry")

    candidates = candidates.join(
        catchment_geom_lookup,
        on="index_right"
    )

    # --------------------------------------------------------
    # True intersection area
    # --------------------------------------------------------
    candidates["intersection_area_m2"] = intersection_area_chunked(
        candidates,
        chunk_size=150_000
    )

    candidates = candidates[
        candidates["intersection_area_m2"] > 0.01
    ].copy()

    candidates["fraction_inside"] = (
        candidates["intersection_area_m2"] / candidates["cell_area_m2"]
    ).clip(0, 1)

    # --------------------------------------------------------
    # Save useful columns
    # --------------------------------------------------------
    keep_cols = [
        "cell_id",
        "row",
        "col",
        "lon",
        "lat",
        "divide_id",
        "vpuid",
        "nexus_id",
        "cell_area_m2",
        "intersection_area_m2",
        "fraction_inside"
    ]

    optional_cols = ["nexus_type", "is_terminal", "terminal_nexus_id"]

    for c in optional_cols:
        if c in candidates.columns and c not in keep_cols:
            keep_cols.append(c)

    df = (
        pd.DataFrame(candidates[keep_cols])
        .drop_duplicates(subset=["divide_id", "cell_id"])
        .sort_values(["divide_id", "cell_id"])
        .reset_index(drop=True)
    )

    # Normalize to weights that sum to 1 for each catchment
    # (kept for reference/QA only -- Notebook 4 re-normalizes over
    #  whichever cells actually have data after dropping NaNs per timestep,
    #  rather than using this pre-baked weight directly)
    df["weight"] = (
        df["fraction_inside"] /
        df.groupby("divide_id")["fraction_inside"].transform("sum")
    )

    df.to_parquet(out_file, index=False)
    parts.append(df)

    print(f"  final rows: {len(df):,}")
    print(f"  catchments hit: {df['divide_id'].nunique():,}")
    print(f"  saved: {out_file}")


# ------------------------------------------------------------
# Combine all selected VPU fractional files
# ------------------------------------------------------------
selected_fractional_crosswalk = (
    pd.concat(parts, ignore_index=True)
    .drop_duplicates(subset=["divide_id", "cell_id"])
    .reset_index(drop=True)
)

selected_fractional_crosswalk["weight"] = (
    selected_fractional_crosswalk["fraction_inside"] /
    selected_fractional_crosswalk.groupby("divide_id")["fraction_inside"].transform("sum")
)

selected_fractional_crosswalk.to_parquet(SELECTED_FRAC_FILE, index=False)

print("\n============================================================")
print("FRACTIONAL CROSSWALK READY \u2705")
print("============================================================")
print(f"rows               : {len(selected_fractional_crosswalk):,}")
print(f"unique catchments  : {selected_fractional_crosswalk['divide_id'].nunique():,}")
print(f"unique MRMS cells  : {selected_fractional_crosswalk['cell_id'].nunique():,}")
print(f"saved file         : {SELECTED_FRAC_FILE}")
print(f"total time         : {(time.time() - t0) / 60:.1f} minutes")

weight_check = selected_fractional_crosswalk.groupby("divide_id")["weight"].sum()
print("\nWeight sum check (should all be ~1.0):")
print(weight_check.describe())

VPUs to process: 1
['03N']

Working on VPU 03N...
  selected catchments: 277
  catchments without center MRMS cell: 2
  candidate MRMS cells: 3,582
  candidate cell/catchment pairs: 4,776
  final rows: 4,776
  catchments hit: 277
  saved: event_mrms_outputs/step3_event_relevant/selected_fractional_crosswalk_cache/selected_fractional_crosswalk_vpu_03N.parquet

FRACTIONAL CROSSWALK READY ✅
rows               : 4,776
unique catchments  : 277
unique MRMS cells  : 2,560
saved file         : event_mrms_outputs/step3_event_relevant/selected_fractional_crosswalk.parquet
total time         : 0.0 minutes

Weight sum check (should all be ~1.0):
count    2.770000e+02
mean     1.000000e+00
std      3.896684e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: weight, dtype: float64


## Save outputs for Notebook 4

Notebook 4 needs: the event manifest (window + recording gage per event), the
long-form event\u2192catchment table, the fractional crosswalk, and the shard
directory the rainfall actually lives in. `fraction_inside` (not the
pre-normalized `weight` column) is what Notebook 4 should use, re-normalizing
per timestep after dropping any cells that come back NaN \u2014 see the data
conventions note in the cell above.

In [25]:
OUT_DIR3 = Path("event_mrms_outputs")
OUT_DIR3.mkdir(exist_ok=True)

manifest_out.to_parquet(OUT_DIR3 / "manifest.parquet", index=False)
event_catchment_windows.to_parquet(OUT_DIR3 / "event_catchment_windows.parquet", index=False)
selected_fractional_crosswalk.to_parquet(OUT_DIR3 / "selected_fractional_crosswalk.parquet", index=False)

with open(OUT_DIR3 / "shard_dir.txt", "w") as f:
    f.write(str(SHARD_DIR.resolve()))

print("Saved to", OUT_DIR3.resolve())
for f in sorted(OUT_DIR3.iterdir()):
    print(" -", f.name, f"({f.stat().st_size / 1e6:.2f} MB)")

Saved to /home/jovyan/Group_Project/event_mrms_outputs
 - event_catchment_windows.parquet (0.26 MB)
 - manifest.parquet (0.19 MB)
 - selected_fractional_crosswalk.parquet (0.20 MB)
 - shard_dir.txt (0.00 MB)
 - step3_event_relevant (0.00 MB)


## Notebook 3 \u2014 done \u2705

- Gage-basin event manifest built: recording gage + 5-day window + upstream
  contributing catchments per event.
- MRMS PrecipRate downloaded via `build_store()` into resumable day shards.
- Fractional MRMS\u2192catchment crosswalk built for exactly the gage-basin
  catchments needed.
- Everything saved to `event_mrms_outputs/` for **Notebook 4**.

**Next: Notebook 4** \u2014 for each event, read the relevant day shards, extract
the fractional-crosswalk cells, area-weight to per-catchment 2-min rainfall
rate (`\u03a3(rate \u00d7 fraction_inside) / \u03a3(fraction_inside)`, re-normalized over
non-NaN cells per timestep), resample to 15-min mean rate, convert to depth
(`rate15 \u00d7 0.25`), and export per-storm NetCDF forcing.